**Career Prediction Model**

In [ ]:
import pandas as pd
import numpy as np

import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

import pickle

In [ ]:
import os

print(os.path.getsize('/content/resume2.csv'))

17060938


In [ ]:
with open('/content/resume2.csv', 'r', encoding='utf-8', errors='ignore') as f:
    for i in range(5):
        print(f.readline())

ID,Resume_str,Resume_html,Category

RES_DATA_SCIENTIST_0001,"Name: Sarah Jenkins

Location: Hyderabad, India

Category: Data Scientist





In [ ]:
df = pd.read_csv(
    '/content/resume2.csv',
    engine='python'
)

In [ ]:
print(df['Category'].dtype)

object


In [ ]:
print(repr(df['Category'].iloc[0]))

'Data Scientist'


In [ ]:
df[['Resume_str','Category']].head()

,Resume_str,Category
0,"Name: Sarah Jenkins\nLocation: Hyderabad, Indi...",Data Scientist
1,"Name: David Miller\nLocation: London, UK\nCate...",Data Scientist
2,Name: Sarah Jenkins\nLocation: Singapore\nCate...,Data Scientist
3,"Name: Sunita Rao\nLocation: New York, USA\nCat...",Data Scientist
4,"Name: Sunita Rao\nLocation: Bangalore, India\n...",Data Scientist


In [ ]:
print(df.shape)
print(df.columns[:10])

(7600, 4)
Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')


In [ ]:
print(df.columns.tolist()[:20])

['ID', 'Resume_str', 'Resume_html', 'Category']


In [ ]:
df = df[['ID', 'Resume_str', 'Resume_html', 'Category']]

In [ ]:
print(df.shape)
print(df.columns)

(7600, 4)
Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')


In [ ]:
#for now no need
df = pd.read_csv(
    '/content/Resume.csv',
    engine='python',
    on_bad_lines='skip'
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Resume.csv'

In [ ]:
df.shape
df.head()

,ID,Resume_str,Resume_html,Category
0,RES_DATA_SCIENTIST_0001,"Name: Sarah Jenkins\nLocation: Hyderabad, Indi...","<div><h1>Sarah Jenkins</h1><p>Hyderabad, India...",Data Scientist
1,RES_DATA_SCIENTIST_0002,"Name: David Miller\nLocation: London, UK\nCate...","<div><h1>David Miller</h1><p>London, UK</p><h2...",Data Scientist
2,RES_DATA_SCIENTIST_0003,Name: Sarah Jenkins\nLocation: Singapore\nCate...,<div><h1>Sarah Jenkins</h1><p>Singapore</p><h2...,Data Scientist
3,RES_DATA_SCIENTIST_0004,"Name: Sunita Rao\nLocation: New York, USA\nCat...","<div><h1>Sunita Rao</h1><p>New York, USA</p><h...",Data Scientist
4,RES_DATA_SCIENTIST_0005,"Name: Sunita Rao\nLocation: Bangalore, India\n...","<div><h1>Sunita Rao</h1><p>Bangalore, India</p...",Data Scientist


In [ ]:
#for now no need
resume_df = pd.read_csv(
    '/content/Resume.csv',
    engine='python'
)

resume_df = resume_df[
    ['ID','Resume_str','Resume_html','Category']
]

FileNotFoundError: [Errno 2] No such file or directory: '/content/Resume.csv'

In [ ]:
df = df[['Resume_str', 'Category']]

In [ ]:
print(df['Category'].value_counts())

Category
Data Scientist                   100
Machine Learning Engineer        100
AI Engineer                      100
Prompt Engineer                  100
NLP Engineer                     100
                                ... 
Hotel Management Professional    100
Chef                             100
Agriculture Specialist           100
Fitness Trainer                  100
Public Relations Executive       100
Name: count, Length: 76, dtype: int64


In [ ]:
print(df.isnull().sum())

Resume_str    0
Category      0
dtype: int64


In [ ]:
#must run
import re

def clean_resume(text):
    text = str(text)

    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'www\S+', ' ', text)

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'\+?\d[\d\s\-]{8,}', ' ', text)

    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.lower().strip()

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['Category_encoded'] = encoder.fit_transform(df['Category'])

print(df[['Category', 'Category_encoded']].head())

         Category  Category_encoded
0  Data Scientist                25
1  Data Scientist                25
2  Data Scientist                25
3  Data Scientist                25
4  Data Scientist                25


In [ ]:
category_mapping = dict(
    zip(encoder.classes_,
        encoder.transform(encoder.classes_))
)

print(category_mapping)

{'AI Engineer': np.int64(0), 'Academic Counselor': np.int64(1), 'Accountant': np.int64(2), 'Advocate': np.int64(3), 'Agriculture Specialist': np.int64(4), 'Android Developer': np.int64(5), 'Automobile Engineer': np.int64(6), 'Aviation Professional': np.int64(7), 'BPO Executive': np.int64(8), 'Backend Developer': np.int64(9), 'Banking Professional': np.int64(10), 'Business Analyst': np.int64(11), 'Business Development Executive': np.int64(12), 'Cabin Crew': np.int64(13), 'Chartered Accountant': np.int64(14), 'Chef': np.int64(15), 'Civil Engineer': np.int64(16), 'Cloud Architect': np.int64(17), 'Cloud Engineer': np.int64(18), 'Computer Vision Engineer': np.int64(19), 'Content Writer': np.int64(20), 'Customer Support Executive': np.int64(21), 'Cybersecurity Analyst': np.int64(22), 'Data Analyst': np.int64(23), 'Data Engineer': np.int64(24), 'Data Scientist': np.int64(25), 'Database Administrator': np.int64(26), 'DevOps Engineer': np.int64(27), 'Digital Marketing Specialist': np.int64(28),

In [ ]:
df['cleaned_resume'] = df['Resume_str'].apply(clean_resume)

In [ ]:
from sklearn.model_selection import train_test_split

X = df['cleaned_resume']
y = df['Category_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC()

model.fit(X_train_tfidf, y_train)

LinearSVC()

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       1.00      1.00      1.00        20
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00        20
          11       1.00      1.00      1.00        20
          12       1.00      1.00      1.00        20
          13       1.00      1.00      1.00        20
          14       1.00      1.00      1.00        20
          15       1.00      1.00      1.00        20
          16       1.00      1.00      1.00        20
          17       1.00    

In [ ]:
import pickle

# Save Model
with open("career_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save TF-IDF Vectorizer
with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Save Label Encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Files saved successfully!")

Files saved successfully!


In [ ]:
import os

print(os.listdir())

['.config', 'label_encoder.pkl', 'tfidf.pkl', 'resume2.csv', 'career_model.pkl', 'sample_data']


In [ ]:
from google.colab import files

files.download("career_model.pkl")
files.download("tfidf.pkl")
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pickle

model = pickle.load(open("career_model.pkl", "rb"))
tfidf = pickle.load(open("tfidf.pkl", "rb"))
encoder = pickle.load(open("label_encoder.pkl", "rb"))

print("Loaded Successfully")

Loaded Successfully


**Skill Extraction**

In [ ]:
import pandas as pd

career_skills_df = pd.read_csv("/content/career_skills_dataset.csv")

career_skills_df.head()

,Career,Required_Skills
0,Data Scientist,"Python,SQL,Statistics,Machine Learning,Pandas,..."
1,AI Engineer,"Python,Deep Learning,TensorFlow,PyTorch,NLP,Co..."
2,Machine Learning Engineer,"Python,C++,Scikit-Learn,PyTorch,MLOps,Kubeflow..."
3,Senior Data Scientist,"Python,R,Advanced Statistical Modeling,A/B Tes..."
4,Junior Data Scientist,"Python,SQL,Data Cleaning,Exploratory Data Anal..."


In [ ]:
all_skills = set()

for skills in career_skills_df['Required_Skills']:

    skill_list = str(skills).split(',')

    for skill in skill_list:
        all_skills.add(skill.strip().lower())

print("Total Skills:", len(all_skills))

Total Skills: 419


In [ ]:
print(list(all_skills)[:20])

['relational databases', 'active directory', 'swift', 'test case design', 'risk assessment', 'healthcare management', 'prompt optimization', 'nlp', 'rust', 'product lifecycle', 'numpy', 'rest apis', 'negotiation', 'cost accounting', 'load balancing', 'agile methodologies', 'apache airflow', 'slas/slos/slis', 'linux', 'information security governance']


In [ ]:
all_skills = {
    skill
    for skill in all_skills
    if len(skill) > 2
}

In [ ]:
import re

def extract_skills(resume_text):

    resume_text = str(resume_text).lower()

    detected_skills = []

    for skill in all_skills:

        pattern = r'\b' + re.escape(skill.lower()) + r'\b'

        if re.search(pattern, resume_text):
            detected_skills.append(skill)

    return sorted(list(set(detected_skills)))

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Resume.csv', on_bad_lines='skip')

In [ ]:
resume_text = df['Resume_str'].iloc[0]

skills = extract_skills(resume_text)

print(skills)

['accounting', 'budgeting', 'client relations', 'conflict resolution', 'data analysis', 'employee relations', 'medical terminology', 'statistics', 'swift', 'team management', 'time management', 'training and development']


In [ ]:
%whos

Variable                Type               Data/Info
----------------------------------------------------
LinearSVC               type               <class 'sklearn.svm._classes.LinearSVC'>
TfidfVectorizer         type               <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
accuracy_score          function           <function accuracy_score at 0x7ba1acab2340>
all_skills              set                {'relational databases', <...>arch', 'quality control'}
career_skills_df        DataFrame                                 Ca<...>\n\n[65 rows x 2 columns]
classification_report   function           <function classification_<...>report at 0x7ba1acab37e0>
df                      DataFrame                      ID           <...>\n[2484 rows x 4 columns]
encoder                 LabelEncoder       LabelEncoder()
extract_skills          function           <function extract_skills at 0x7ba1ab781e40>
internship_df           DataFrame                               Care<...>n\n[282 ro

In [ ]:
resume_text = df['Resume_str'].iloc[0]

cleaned = clean_resume(resume_text)

vector = tfidf.transform([cleaned])

prediction = model.predict(vector)

predicted_career = encoder.inverse_transform(prediction)[0]

print("Predicted Career:", predicted_career)

Predicted Career: HR


In [ ]:
row = career_skills_df[
    career_skills_df['Career'] == predicted_career
]

required_skills = row.iloc[0]['Required_Skills']

required_skills = [
    skill.strip().lower()
    for skill in required_skills.split(',')
]

print(required_skills)

['recruitment', 'talent acquisition', 'employee relations', 'payroll management', 'hrms', 'performance management', 'labor laws', 'onboarding', 'training and development', 'conflict resolution', 'compensation and benefits', 'communication']


In [ ]:
missing_skills = list(
    set(required_skills) - set(skills)
)

print("Missing Skills:")
print(missing_skills)

Missing Skills:
['compensation and benefits', 'hrms', 'payroll management', 'communication', 'labor laws', 'performance management', 'recruitment', 'talent acquisition', 'onboarding']


In [ ]:
print("\nPredicted Career:", predicted_career)

print("\nDetected Skills:")
for skill in skills:
    print("-", skill)

print("\nMissing Skills:")
for skill in missing_skills:
    print("-", skill)


Predicted Career: HR

Detected Skills:
- accounting
- budgeting
- client relations
- conflict resolution
- data analysis
- employee relations
- medical terminology
- statistics
- swift
- team management
- time management
- training and development

Missing Skills:
- compensation and benefits
- hrms
- payroll management
- communication
- labor laws
- performance management
- recruitment
- talent acquisition
- onboarding


**COURSE RECOMMENDATION**

In [ ]:
course_df = pd.read_csv("/content/skill_course_dataset.csv")

course_df.head()

,Skill,Course,Platform,Difficulty,Duration
0,Python,Python for Data Science and AI,Coursera,Beginner,4 weeks
1,SQL,Learn SQL Basics for Data Science,Coursera,Beginner,5 weeks
2,Machine Learning,Machine Learning Specialization,Coursera,Intermediate,12 weeks
3,Deep Learning,Deep Learning Specialization,Coursera,Advanced,16 weeks
4,Data Visualization,Data Visualization with Tableau,Udemy,Beginner,6 weeks


In [ ]:
print(course_df.columns)

Index(['Skill', 'Course', 'Platform', 'Difficulty', 'Duration'], dtype='object')


In [ ]:
def recommend_courses(missing_skills, course_df):

    recommended_courses = []

    for skill in missing_skills:

        matches = course_df[
            course_df['Skill'].str.lower() == skill.lower()
        ]

        if not matches.empty:

            recommended_courses.extend(
                matches['Course'].tolist()
            )

    return list(set(recommended_courses))

In [ ]:
courses = recommend_courses(
    missing_skills,
    course_df
)

print(courses)

['Employee Onboarding Strategies', 'Recruiting Foundations', 'Performance Management Foundations']


In [ ]:
print("\nRecommended Courses:\n")

for i, course in enumerate(courses, 1):
    print(f"{i}. {course}")


Recommended Courses:

1. Employee Onboarding Strategies
2. Recruiting Foundations
3. Performance Management Foundations


In [ ]:
for skill in missing_skills:

    matches = course_df[
        course_df['Skill'].str.lower() == skill.lower()
    ]

    if not matches.empty:

        print(f"\nSkill: {skill}")

        for course in matches['Course']:
            print("  ->", course)


Skill: performance management
  -> Performance Management Foundations

Skill: talent acquisition
  -> Recruiting Foundations

Skill: onboarding
  -> Employee Onboarding Strategies


**Internship Recommendation**

In [ ]:
internship_df = pd.read_csv("/content/internship_dataset.csv")

internship_df.head()

,Career,Internship_Role,Required_Skills
0,HR,HR Operations Intern,"HRMS, Excel, Employee Onboarding, Documentatio..."
1,HR,Talent Acquisition Intern,"Sourcing, Screening, ATS, Cold Calling, Linked..."
2,HR,Employee Engagement Intern,"Event Planning, Internal Communication, Feedba..."
3,HR,Compensation and Benefits Intern,"Payroll Processing, Excel, Compliance, Data En..."
4,HR,HR Analytics Intern,"Power BI, Excel, Data Cleaning, Dashboarding, ..."


In [ ]:
print(internship_df.columns)

Index(['Career', 'Internship_Role', 'Required_Skills'], dtype='object')


In [ ]:
career_internships = internship_df[
    internship_df['Career'] == predicted_career
]

career_internships

,Career,Internship_Role,Required_Skills
0,HR,HR Operations Intern,"HRMS, Excel, Employee Onboarding, Documentatio..."
1,HR,Talent Acquisition Intern,"Sourcing, Screening, ATS, Cold Calling, Linked..."
2,HR,Employee Engagement Intern,"Event Planning, Internal Communication, Feedba..."
3,HR,Compensation and Benefits Intern,"Payroll Processing, Excel, Compliance, Data En..."
4,HR,HR Analytics Intern,"Power BI, Excel, Data Cleaning, Dashboarding, ..."
5,HR,Employer Branding Intern,"Content Writing, Social Media Marketing, Canva..."
6,HR,Technical Recruiter Intern,"IT Skills Screening, LinkedIn Sourcing, ATS, T..."


In [ ]:
def recommend_internships(user_skills, predicted_career, internship_df):

    user_skills = [s.lower().strip() for s in user_skills]

    recommendations = []

    career_rows = internship_df[
        internship_df['Career'] == predicted_career
    ]

    for _, row in career_rows.iterrows():

        required = [
            s.strip().lower()
            for s in row['Required_Skills'].split(',')
        ]

        score = len(
    set(user_skills).intersection(required)
)

        if predicted_career == row['Career']:
            score += 2

        recommendations.append(
            (row['Internship_Role'], score)
        )

    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations[:5]

In [ ]:
recommended_internships = recommend_internships(
    skills,
    predicted_career,
    internship_df
)

print(recommended_internships)

[('HR Operations Intern', 2), ('Talent Acquisition Intern', 2), ('Employee Engagement Intern', 2), ('Compensation and Benefits Intern', 2), ('HR Analytics Intern', 2)]


In [ ]:
print("\nRecommended Internships:\n")

for internship, score in recommended_internships:

    print(
        f"{internship} (Skill Match Score: {score})"
    )


Recommended Internships:

HR Operations Intern (Skill Match Score: 2)
Talent Acquisition Intern (Skill Match Score: 2)
Employee Engagement Intern (Skill Match Score: 2)
Compensation and Benefits Intern (Skill Match Score: 2)
HR Analytics Intern (Skill Match Score: 2)


In [ ]:
for _, row in internship_df[internship_df['Career'] == 'HR'].iterrows():

    required = [
        s.strip().lower()
        for s in row['Required_Skills'].split(',')
    ]

    common = list(
        set(skills).intersection(required)
    )

    print("\n", row['Internship_Role'])
    print("Common Skills:", common)


 HR Operations Intern
Common Skills: []

 Talent Acquisition Intern
Common Skills: []

 Employee Engagement Intern
Common Skills: []

 Compensation and Benefits Intern
Common Skills: []

 HR Analytics Intern
Common Skills: []

 Employer Branding Intern
Common Skills: []

 Technical Recruiter Intern
Common Skills: []


In [ ]:
print(skills)
print(predicted_career)

['accounting', 'budgeting', 'client relations', 'conflict resolution', 'data analysis', 'employee relations', 'medical terminology', 'statistics', 'swift', 'team management', 'time management', 'training and development']
HR


In [ ]:
readiness_score = round(
    (len(skills) /
     len(required_skills)) * 100,
    2
)

readiness_score = min(readiness_score, 100)

print(
    f"Career Readiness Score: {readiness_score}%"
)

Career Readiness Score: 100.0%


**ROAD MAP**

In [ ]:
project_dict = {

    "HR": "Build an Employee Management System",

    "INFORMATION-TECHNOLOGY": "Build a Full Stack Web Application",

    "FINANCE": "Create a Financial Analysis Dashboard",

    "ACCOUNTANT": "Develop an Accounting Automation Tool",

    "DESIGNER": "Create a Complete UI/UX Case Study",

    "DATA SCIENTIST": "Build an End-to-End Machine Learning Project"
}


def generate_roadmap(predicted_career, missing_skills):

    roadmap = []
    step = 1

    # Learn missing skills
    for skill in missing_skills:
        roadmap.append(f"Step {step}: Learn {skill.title()}")
        step += 1

    # Career-specific project
    project = project_dict.get(
        predicted_career,
        f"Build a Project in {predicted_career}"
    )

    roadmap.append(f"Step {step}: {project}")
    step += 1

    # Internship
    roadmap.append(
        f"Step {step}: Apply for {predicted_career} Internships"
    )
    step += 1

    # Final goal
    roadmap.append(
        f"Step {step}: Become a {predicted_career} Professional"
    )

    return roadmap

In [ ]:
roadmap = generate_roadmap(
    predicted_career,
    missing_skills
)

In [ ]:
print("\nCAREER ROADMAP\n")

for item in roadmap:
    print(item)


CAREER ROADMAP

Step 1: Learn Compensation And Benefits
Step 2: Learn Hrms
Step 3: Learn Payroll Management
Step 4: Learn Communication
Step 5: Learn Labor Laws
Step 6: Learn Performance Management
Step 7: Learn Recruitment
Step 8: Learn Talent Acquisition
Step 9: Learn Onboarding
Step 10: Build an Employee Management System
Step 11: Apply for HR Internships
Step 12: Become a HR Professional


In [ ]:
roadmap = generate_roadmap(
    predicted_career,
    missing_skills
)

for item in roadmap:
    print(item)

Step 1: Learn Compensation And Benefits
Step 2: Learn Hrms
Step 3: Learn Payroll Management
Step 4: Learn Communication
Step 5: Learn Labor Laws
Step 6: Learn Performance Management
Step 7: Learn Recruitment
Step 8: Learn Talent Acquisition
Step 9: Learn Onboarding
Step 10: Build an Employee Management System
Step 11: Apply for HR Internships
Step 12: Become a HR Professional


In [ ]:
!pip install streamlit
import streamlit as st

st.title("🎯 AI Career Mentor")

st.write("Upload your resume and get career recommendations.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 84.0 MB/s eta 0:00:00


2026-06-14 13:00:34.472 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 13:00:34.662 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-14 13:00:34.663 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 13:00:34.664 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 13:00:34.666 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 13:00:34.667 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 13:00:34.667 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
import numpy
import sklearn

print(numpy.__version__)
print(sklearn.__version__)

2.0.2
1.6.1
